<a href="https://colab.research.google.com/github/firdoushkhilji/firdoushkhilji-7006SCN_FK_16943920/blob/task2-branch/Task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
#import and start pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NHS_Prescription_Cost_Prediction") \
    .getOrCreate()

print("SparkSession started successfully!")

SparkSession started successfully!


In [21]:
#mount google drive and load dataset
from google.colab import drive
drive.mount('/content/drive')

#READ CSV FILE USING SPARK
df = spark.read.csv('/content/drive/MyDrive/TASK_DATASET/epd_snomed_202603.csv', header=True, inferSchema=True)

print('Dataset Loaded Successfully!')
print(f'no. of record: {df.count():,}')
print(f'no.of columns: {len(df.columns)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset Loaded Successfully!
no. of record: 18,364,409
no.of columns: 27


In [22]:
#check how many parts the data is split into
print(f'no. of partitions: {df.rdd.getNumPartitions()}')

no. of partitions: 58


In [23]:
#split the partition into 8 section instead
df = df.repartition(8)
print(f'no. of partitions: {df.rdd.getNumPartitions()}')

no. of partitions: 8


In [24]:
#check partition size
from pyspark.sql.functions import spark_partition_id, count

df.groupBy(spark_partition_id().alias("partition_id")) \
  .count() \
  .orderBy("partition_id") \
  .show()

+------------+-------+
|partition_id|  count|
+------------+-------+
|           0|2295550|
|           1|2295550|
|           2|2295554|
|           3|2295553|
|           4|2295552|
|           5|2295550|
|           6|2295552|
|           7|2295548|
+------------+-------+



In [25]:
#cleaning the dataset - drop useless columns

#drop columns that are not required for prediction of cost of prescription
drop_columns = ['ADDRESS_1', 'ADDRESS_2', 'ADDRESS_3', 'ADDRESS_4',
    'UNIDENTIFIED', 'YEAR_MONTH',
    'BNF_CHEMICAL_SUBSTANCE_CODE', 'BNF_PRESENTATION_CODE',
    'PRACTICE_CODE', 'PCO_CODE', 'ICB_CODE',
    'REGIONAL_OFFICE_CODE']

df = df.drop(*drop_columns)
print('columns after dropping - ')
print(df.columns)

columns after dropping - 
['REGIONAL_OFFICE_NAME', 'ICB_NAME', 'PCO_NAME', 'PRACTICE_NAME', 'POSTCODE', 'BNF_CHEMICAL_SUBSTANCE', 'BNF_PRESENTATION_NAME', 'BNF_CHAPTER_PLUS_CODE', 'QUANTITY', 'ITEMS', 'TOTAL_QUANTITY', 'ADQ_USAGE', 'NIC', 'ACTUAL_COST', 'SNOMED_CODE']


In [26]:
print(f'no.of columns: {len(df.columns)}')

no.of columns: 15


In [27]:
#fix empty value
from pyspark.sql.functions import col, isnan, when, sum as spark_sum

#check missing values
print('Missing values as per columns- ')
df.select([
    spark_sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
]).show()

#drop rows with missing ACTUAL_COST value - because a row with no cost is useless for training
df = df.dropna(subset=['ACTUAL_COST'])

# Fill missing POSTCODE
df = df.fillna({'POSTCODE': 'UNKNOWN'})

# fill remaining null values with 0
df = df.fillna({
    'QUANTITY': 0.0,
    'ITEMS': 0,
    'TOTAL_QUANTITY': 0.0,
    'ADQ_USAGE': 0.0,
    'NIC': 0.0,
    'SNOMED_CODE': 0
})

# Verify no missing values are remaining
print("Missing values AFTER cleaning:")
df.select([
    spark_sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
]).show()

#print rows left after cleaning
print(f"No. of Records after cleaning: {df.count():,}")

Missing values as per columns- 
+--------------------+--------+--------+-------------+--------+----------------------+---------------------+---------------------+--------+-----+--------------+---------+---+-----------+-----------+
|REGIONAL_OFFICE_NAME|ICB_NAME|PCO_NAME|PRACTICE_NAME|POSTCODE|BNF_CHEMICAL_SUBSTANCE|BNF_PRESENTATION_NAME|BNF_CHAPTER_PLUS_CODE|QUANTITY|ITEMS|TOTAL_QUANTITY|ADQ_USAGE|NIC|ACTUAL_COST|SNOMED_CODE|
+--------------------+--------+--------+-------------+--------+----------------------+---------------------+---------------------+--------+-----+--------------+---------+---+-----------+-----------+
|                   0|       0|       0|            0|    6793|                     0|                    0|                    0|       0|    0|             0|        0|  0|          0|      11449|
+--------------------+--------+--------+-------------+--------+----------------------+---------------------+---------------------+--------+-----+--------------+---------+--

In [28]:
# for classification - creating a column
#HIGH_COST - gives 1 or 0(binary)
# 1 = prescription costs more than £50, = £50 and 0 is costs less than £50
from pyspark.sql.functions import when

df = df.withColumn('HIGH_COST',
    when(col('ACTUAL_COST') > 50, 1).otherwise(0)
)
print("HIGH_COST column created!")
df.groupBy('HIGH_COST').count().show()

HIGH_COST column created!
+---------+--------+
|HIGH_COST|   count|
+---------+--------+
|        1| 3490390|
|        0|14874019|
+---------+--------+



In [29]:
# Feature Engineering — Log Transforms
# ACTUAL_COST and NIC are heavily right-skewed
# Most prescriptions are cheap but a few are very expensive
# Log transform makes the distribution more normal for ML models
# log1p = log(1+x) safely handles zero values

from pyspark.sql.functions import log1p

df = df.withColumn('LOG_NIC', log1p(col('NIC')))
df = df.withColumn('LOG_QUANTITY', log1p(col('QUANTITY')))
df = df.withColumn('LOG_TOTAL_QUANTITY', log1p(col('TOTAL_QUANTITY')))

print("Log transforms applied!")
print(f"Columns after Log transformation: {len(df.columns)}")

Log transforms applied!
Columns after Log transformation: 19


In [30]:
#Encoding
#import stringindexer or converting categorical text columns into numeric indices
from pyspark.ml.feature import StringIndexer

#columns to be encoded for model training
columns_to_encoded = ['REGIONAL_OFFICE_NAME',
    'ICB_NAME',
    'PCO_NAME',
    'PRACTICE_NAME',
    'BNF_CHEMICAL_SUBSTANCE',
    'BNF_PRESENTATION_NAME',
    'BNF_CHAPTER_PLUS_CODE']

#create string index for each column
indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=col + "_IDX",
        handleInvalid="keep" #unseen or null values are assigned a separate index
    )
    for col in columns_to_encoded
]

print(f"{len(indexers)} columns encoded!")

7 columns encoded!


In [31]:
# ONE HOT ENCODING
# StringIndexer gives numbers like 0,1,2,3 for categories
# Problem: model thinks category 3 is "bigger" than category 1
# OneHotEncoder converts to binary vectors — no false ordering
# e.g. LONDON=[1,0,0], MIDLANDS=[0,1,0], NORTH=[0,0,1]

from pyspark.ml.feature import OneHotEncoder

encoder = OneHotEncoder(
    inputCols=[c + "_IDX" for c in columns_to_encoded],
    outputCols=[c + "_OHE" for c in columns_to_encoded],
    handleInvalid="keep"
)
print("OneHotEncoder created!")

OneHotEncoder created!


In [32]:
#vector assembler
from pyspark.ml.feature import VectorAssembler
#combine all feature columns for ML - fit all columns used for prediction into one
features = [ 'QUANTITY', # How many units were prescribed
           'ITEMS', # Number of prescription items
            'TOTAL_QUANTITY', # Total quantity across all items
            'ADQ_USAGE', # Average Daily Quantity usage
            'NIC', # Net Ingredient Cost (closely related to actual cost)
            'SNOMED_CODE', # Clinical code for the drug
            'LOG_NIC',
            'LOG_QUANTITY',
            'LOG_TOTAL_QUANTITY',
            'REGIONAL_OFFICE_NAME_OHE', # Region encoded as number(changed from _IDX)
            'ICB_NAME_OHE', # Integrated Care Board encoded as number(changed from _IDX)
            'PCO_NAME_OHE', # Primary Care Organisation encoded as number(changed from _IDX)
            'PRACTICE_NAME_OHE', # GP Practice encoded as number(changed from _IDX)
            'BNF_CHEMICAL_SUBSTANCE_OHE', # Drug name encoded as number(changed from _IDX)
            'BNF_PRESENTATION_NAME_OHE', # Drug form encoded as number(changed from _IDX)
            'BNF_CHAPTER_PLUS_CODE_OHE' # Drug category encoded as number (changed from _IDX)
            ]

#using assembler to combine all features into one vector
assembler = VectorAssembler(inputCols= features,
                            outputCol="features", # Output column name
                            handleInvalid="keep" # to keep rows with invalid values instead of crashing
                             )

print("VectorAssembler created successfully!")

VectorAssembler created successfully!


In [33]:
#scaling
#StandardScaler rescales everything to the same range for fair comparison
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",           # Takes features vector from previous cell
    outputCol="scaled_features",   # gives rescaled version
    withStd=True,                  # with std means divide by standard deviation — normalises spread
    withMean=False                 # Don't subtract mean — keeps sparse data efficient
)

print("StandardScaler created successfully!")

StandardScaler created successfully!


In [34]:
#BUILDING A PIPELINE
#a pipeline helps automate a process, that is same for both train and test data.
#first encode, then aseemble and then scale data.
from pyspark.ml import Pipeline

pipeline_stages = indexers + [encoder, assembler, scaler] #encoding --> assemble --> scale

pipeline = Pipeline(stages=pipeline_stages)

print(f"Pipeline built with {len(pipeline_stages)} stages!")
print("Stages in order:")
for i, stage in enumerate(pipeline_stages):
    print(f"  Stage {i+1}: {type(stage).__name__}")

Pipeline built with 10 stages!
Stages in order:
  Stage 1: StringIndexer
  Stage 2: StringIndexer
  Stage 3: StringIndexer
  Stage 4: StringIndexer
  Stage 5: StringIndexer
  Stage 6: StringIndexer
  Stage 7: StringIndexer
  Stage 8: OneHotEncoder
  Stage 9: VectorAssembler
  Stage 10: StandardScaler


In [36]:
#fit and transform data
# pipeline.transform() — applies all transformations to produce final dataset
# This is where all the actual work happens
# fit() can take several minutes on 18 million rows — this is normal

print("Fitting pipeline to data -")
pipeline_model = pipeline.fit(df)  # Learn from data

# applies all transformations to produce final dataset using transform()
df_transformed = pipeline_model.transform(df)

print("Pipeline fitted and data transformed successfully!")
print(f"Total columns after transformation: {len(df_transformed.columns)}")

# Preview the key output columns
# features = raw assembled vector
# scaled_features = normalised vector (this goes into ML models)
# ACTUAL_COST = what we are trying to predict
df_transformed.select('features', 'scaled_features', 'ACTUAL_COST').show(5)


Fitting pipeline to data -
Pipeline fitted and data transformed successfully!
Total columns after transformation: 35
+--------------------+--------------------+-----------+
|            features|     scaled_features|ACTUAL_COST|
+--------------------+--------------------+-----------+
|(32303,[0,1,2,4,5...|(32303,[0,1,2,4,5...|    5.70959|
|(32303,[0,1,2,3,4...|(32303,[0,1,2,3,4...|   19.03198|
|(32303,[0,1,2,3,4...|(32303,[0,1,2,3,4...|   29.14196|
|(32303,[0,1,2,4,5...|(32303,[0,1,2,4,5...|    3.63064|
|(32303,[0,1,2,3,4...|(32303,[0,1,2,3,4...|    0.99609|
+--------------------+--------------------+-----------+
only showing top 5 rows


In [38]:
#TRAIN & TEST DATA
#80% for training and 20% for testing
train_df, test_df = df_transformed.randomSplit([0.8, 0.2],  # 80% train, 20% test
                                                seed=42      # Random seed for reproducibility
                                              )

print(f'no. of record: {df_transformed.count():,}')
print(f"no. of Training records: {train_df.count():,}")
print(f"no. of Testing records:  {test_df.count():,}")

no. of record: 18,364,409
no. of Training records: 14,690,334
no. of Testing records:  3,674,075


In [39]:
# SAVE TRAIN/TEST DATA
# Parquet is a compressed column-based format, faster and easier to load than csv
train_df.write.parquet(
    '/content/drive/MyDrive/TASK_DATASET/train_data.parquet',
    mode='overwrite' #replaces already exisiting file(if any)
)
test_df.write.parquet(
    '/content/drive/MyDrive/TASK_DATASET/test_data.parquet',
    mode='overwrite'#replaces already exisiting file(if any)
)

print("Train and test data saved as Parquet!")

Train and test data saved as Parquet!


## Data Dictionary

| Column | Type | Description | Used In ML |
|--------|------|-------------|------------|
| REGIONAL_OFFICE_NAME | String | NHS England region | Encoded |
| ICB_NAME | String | Integrated Care Board | Encoded |
| PCO_NAME | String | Primary Care Organisation | Encoded |
| PRACTICE_NAME | String | GP Surgery name | Encoded |
| POSTCODE | String | GP Surgery postcode | Dropped after encoding |
| BNF_CHEMICAL_SUBSTANCE | String | Drug name | Encoded |
| BNF_PRESENTATION_NAME | String | Drug form/strength | Encoded |
| BNF_CHAPTER_PLUS_CODE | String | Drug category | Encoded |
| QUANTITY | Float | Units prescribed | Direct |
| ITEMS | Integer | Number of prescriptions | Direct |
| TOTAL_QUANTITY | Float | Total units | Direct |
| ADQ_USAGE | Float | Average Daily Quantity | Direct |
| NIC | Float | Net Ingredient Cost | Direct |
| ACTUAL_COST | Float | TARGET - Regression | Target |
| HIGH_COST | Integer | TARGET - Classification | Target |
| LOG_NIC | Float | Log transform of NIC | Engineered |
| LOG_QUANTITY | Float | Log transform of Quantity | Engineered |
| SNOMED_CODE | Long | Clinical drug code | Direct |

## PySpark Pipeline Architecture
```
RAW DATA (18.3M rows, 27 columns)
         ↓
[STAGE 1-7]  StringIndexer × 7
             Text columns → Numeric indices
             e.g. "LONDON" → 1.0
         ↓
[STAGE 8]    OneHotEncoder
             Numeric indices → Binary vectors
             e.g. 1.0 → [0,1,0,0,0,0,0,0]
         ↓
[STAGE 9]    VectorAssembler
             16 columns → 1 feature vector
             e.g. [224.0, 1, 44.67, ..., 0,1,0]
         ↓
[STAGE 10]   StandardScaler
             Normalise all values to same scale
             e.g. [0.23, 0.01, 0.87, ...]
         ↓
TRANSFORMED DATA — ready for ML models
```